# CyberLey — Análisis de encuesta externa de ciberseguridad

Este notebook analiza los datos históricos recopilados mediante una encuesta externa sobre hábitos digitales y ciberseguridad.

## Objetivo general

Aplicar un proceso de ciencia de datos sobre el archivo CSV de la encuesta:

1. cargar los datos;
2. revisar estructura y calidad;
3. limpiar columnas innecesarias;
4. normalizar respuestas;
5. calcular estadísticas descriptivas;
6. generar gráficas;
7. obtener conclusiones;
8. exportar un CSV limpio.

> Este análisis complementa el sistema CyberLey. La encuesta interna del programa sigue funcionando aparte con Supabase.


In [ ]:
# IMPORTAR LIBRERÍAS

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 14

print("Librerías cargadas correctamente.")


## 1. Cargar el archivo CSV

Este notebook puede leer:

- `Encuesta Ciberseguridad.csv`, si estás usando el archivo original.
- `encuesta_ciberseguridad_limpia.csv`, si ya lo descargaste desde el módulo de importación de CyberLey.

Colocá el archivo dentro de la carpeta `Notebook`.


In [ ]:
# CARGAR ARCHIVO CSV

RUTA_NOTEBOOK = Path.cwd()

opciones_archivo = [
    RUTA_NOTEBOOK / "encuesta_ciberseguridad_limpia.csv",
    RUTA_NOTEBOOK / "Encuesta Ciberseguridad.csv"
]

RUTA_CSV = None

for ruta in opciones_archivo:
    if ruta.exists():
        RUTA_CSV = ruta
        break

if RUTA_CSV is None:
    raise FileNotFoundError(
        "No se encontró el archivo CSV. "
        "Coloca 'Encuesta Ciberseguridad.csv' o "
        "'encuesta_ciberseguridad_limpia.csv' dentro de la carpeta Notebook."
    )

try:
    df_original = pd.read_csv(RUTA_CSV)

except UnicodeDecodeError:
    df_original = pd.read_csv(
        RUTA_CSV,
        encoding="latin-1"
    )

print(f"Archivo cargado: {RUTA_CSV.name}")
print(f"Filas: {df_original.shape[0]}")
print(f"Columnas: {df_original.shape[1]}")

display(df_original.head())


## 2. Revisión inicial de los datos

Antes de limpiar, se revisa:

- cantidad de filas;
- cantidad de columnas;
- nombres de columnas;
- valores faltantes;
- tipos de datos.


In [ ]:
# INFORMACIÓN GENERAL DEL DATASET

print("Dimensiones del dataset:")
print(df_original.shape)

print("\nColumnas disponibles:")
for i, columna in enumerate(df_original.columns):
    print(f"{i}. {columna}")

print("\nValores faltantes por columna:")
display(
    df_original
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "Columna", 0: "Valores faltantes"})
)

print("\nTipos de datos:")
display(
    df_original
    .dtypes
    .reset_index()
    .rename(columns={"index": "Columna", 0: "Tipo de dato"})
)


## 3. Limpieza y normalización

Se aplican las siguientes acciones:

- eliminar columnas `Unnamed`;
- renombrar columnas con nombres más cortos;
- limpiar espacios innecesarios;
- reemplazar vacíos válidos por `No aplica`;
- normalizar categorías como `Rauter` y `Wifi`;
- convertir escalas numéricas.


In [ ]:
# LIMPIEZA DEL DATASET

def limpiar_texto(valor):
    if pd.isna(valor):
        return valor
    return " ".join(str(valor).strip().split())


def limpiar_dataset_encuesta(df_original):
    df = df_original.copy()

    columnas_unnamed = [
        columna
        for columna in df.columns
        if str(columna).startswith("Unnamed:")
    ]

    df = df.drop(columns=columnas_unnamed, errors="ignore")

    columnas_limpias_esperadas = {
        "fecha_respuesta",
        "usa_nube",
        "plataforma_nube",
        "contenido_nube",
        "nivel_conocimiento",
        "manejo_ciberseguridad",
        "frecuencia_info_seguridad",
        "reconoce_phishing",
        "identifica_herramientas_seguridad",
        "estado_antivirus",
        "tipo_conexion",
        "estabilidad_conexion",
        "frecuencia_fallas_internet",
        "cambio_contrasenas_anual",
        "reutiliza_contrasenas",
        "importancia_actualizar_contrasenas"
    }

    if not columnas_limpias_esperadas.issubset(set(df.columns)):
        if len(df.columns) < 16:
            raise ValueError("El CSV tiene menos columnas de las esperadas para este análisis.")

        nombres_nuevos = {
            df.columns[0]: "fecha_respuesta",
            df.columns[1]: "usa_nube",
            df.columns[2]: "plataforma_nube",
            df.columns[3]: "contenido_nube",
            df.columns[4]: "nivel_conocimiento",
            df.columns[5]: "manejo_ciberseguridad",
            df.columns[6]: "frecuencia_info_seguridad",
            df.columns[7]: "reconoce_phishing",
            df.columns[8]: "identifica_herramientas_seguridad",
            df.columns[9]: "estado_antivirus",
            df.columns[10]: "tipo_conexion",
            df.columns[11]: "estabilidad_conexion",
            df.columns[12]: "frecuencia_fallas_internet",
            df.columns[13]: "cambio_contrasenas_anual",
            df.columns[14]: "reutiliza_contrasenas",
            df.columns[15]: "importancia_actualizar_contrasenas"
        }

        df = df.rename(columns=nombres_nuevos)

    columnas_texto = df.select_dtypes(include="object").columns

    for columna in columnas_texto:
        df[columna] = df[columna].apply(limpiar_texto)

    if "plataforma_nube" in df.columns:
        df["plataforma_nube"] = df["plataforma_nube"].fillna("No aplica")

    if "contenido_nube" in df.columns:
        df["contenido_nube"] = df["contenido_nube"].fillna("No aplica")

    if "importancia_actualizar_contrasenas" in df.columns:
        df["importancia_actualizar_contrasenas"] = (
            df["importancia_actualizar_contrasenas"]
            .fillna("Sin respuesta")
        )

    if "tipo_conexion" in df.columns:
        equivalencias_conexion = {
            "Rauter": "Router",
            "Wifi": "Wi-Fi",
            "WiFi": "Wi-Fi",
            "wifi": "Wi-Fi",
            "ADSL": "ADSL",
            "Satelital (Línea de Abonado Digital Asimétrica)": "Satelital"
        }

        df["tipo_conexion"] = df["tipo_conexion"].replace(equivalencias_conexion)

    equivalencias_si_no = {
        "Si": "Sí",
        "si": "Sí",
        "SI": "Sí",
        "Sí": "Sí",
        "No": "No",
        "no": "No"
    }

    for columna in ["usa_nube", "reutiliza_contrasenas"]:
        if columna in df.columns:
            df[columna] = df[columna].replace(equivalencias_si_no)

    for columna in [
        "manejo_ciberseguridad",
        "estabilidad_conexion",
        "importancia_actualizar_contrasenas"
    ]:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce")

    if "fecha_respuesta" in df.columns:
        df["fecha_respuesta"] = (
            df["fecha_respuesta"]
            .astype(str)
            .str.replace(" GMT-6", "", regex=False)
        )

    return df


df_limpio = limpiar_dataset_encuesta(df_original)

print("Dataset limpio generado.")
print(f"Filas: {df_limpio.shape[0]}")
print(f"Columnas: {df_limpio.shape[1]}")

display(df_limpio.head())


## 4. Comparación antes y después de la limpieza

In [ ]:
# COMPARACIÓN DEL PROCESO DE LIMPIEZA

resumen_limpieza = pd.DataFrame({
    "Indicador": [
        "Filas originales",
        "Columnas originales",
        "Valores faltantes originales",
        "Filas limpias",
        "Columnas limpias",
        "Valores faltantes después de limpieza"
    ],
    "Valor": [
        df_original.shape[0],
        df_original.shape[1],
        int(df_original.isna().sum().sum()),
        df_limpio.shape[0],
        df_limpio.shape[1],
        int(df_limpio.isna().sum().sum())
    ]
})

display(resumen_limpieza)


## 5. Estadísticas descriptivas

Se calculan métricas generales para entender el comportamiento de las personas encuestadas.


In [ ]:
# ESTADÍSTICAS DESCRIPTIVAS

total_respuestas = len(df_limpio)

print(f"Total de respuestas analizadas: {total_respuestas}")

for columna, titulo in [
    ("usa_nube", "Uso de almacenamiento en la nube"),
    ("nivel_conocimiento", "Nivel de conocimiento en ciberseguridad"),
    ("reconoce_phishing", "Reconocimiento de phishing"),
    ("estado_antivirus", "Estado del antivirus"),
]:
    if columna in df_limpio.columns:
        print(f"\n{titulo}:")
        display(df_limpio[columna].value_counts(dropna=False))

if "manejo_ciberseguridad" in df_limpio.columns:
    print("\nPromedio de manejo de ciberseguridad:")
    display(df_limpio["manejo_ciberseguridad"].mean())

if "estabilidad_conexion" in df_limpio.columns:
    print("\nPromedio de estabilidad de conexión:")
    display(df_limpio["estabilidad_conexion"].mean())


# Gráficas

A continuación se generan visualizaciones para explicar los resultados de la encuesta externa.


## Gráfica 1. Uso de almacenamiento en la nube

In [ ]:
# GRÁFICA 1: USO DE ALMACENAMIENTO EN LA NUBE

conteo = df_limpio["usa_nube"].value_counts().reset_index()
conteo.columns = ["Respuesta", "Cantidad"]

display(conteo)

plt.figure(figsize=(8, 5))
plt.bar(conteo["Respuesta"], conteo["Cantidad"])
plt.title("Uso de plataformas de almacenamiento en la nube")
plt.xlabel("Respuesta")
plt.ylabel("Cantidad de personas")
plt.tight_layout()
plt.savefig("grafica_1_uso_nube.png", bbox_inches="tight")
plt.show()


## Gráfica 2. Plataformas de nube más utilizadas

In [ ]:
# GRÁFICA 2: PLATAFORMAS DE NUBE MÁS UTILIZADAS

datos = df_limpio[df_limpio["plataforma_nube"] != "No aplica"]
conteo = datos["plataforma_nube"].value_counts().reset_index()
conteo.columns = ["Plataforma", "Cantidad"]

display(conteo)

plt.figure(figsize=(10, 6))
plt.barh(conteo["Plataforma"], conteo["Cantidad"])
plt.title("Plataformas de nube más utilizadas")
plt.xlabel("Cantidad de personas")
plt.ylabel("Plataforma")
plt.tight_layout()
plt.savefig("grafica_2_plataformas_nube.png", bbox_inches="tight")
plt.show()


## Gráfica 3. Nivel de conocimiento en ciberseguridad

In [ ]:
# GRÁFICA 3: NIVEL DE CONOCIMIENTO EN CIBERSEGURIDAD

conteo = df_limpio["nivel_conocimiento"].value_counts().reset_index()
conteo.columns = ["Nivel de conocimiento", "Cantidad"]

display(conteo)

plt.figure(figsize=(9, 5))
plt.bar(conteo["Nivel de conocimiento"], conteo["Cantidad"])
plt.title("Nivel de conocimiento sobre ciberseguridad")
plt.xlabel("Nivel declarado")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("grafica_3_nivel_conocimiento.png", bbox_inches="tight")
plt.show()


## Gráfica 4. Reconocimiento de phishing

In [ ]:
# GRÁFICA 4: RECONOCIMIENTO DE PHISHING

conteo = df_limpio["reconoce_phishing"].value_counts().reset_index()
conteo.columns = ["Respuesta", "Cantidad"]

display(conteo)

plt.figure(figsize=(9, 5))
plt.bar(conteo["Respuesta"], conteo["Cantidad"])
plt.title("Capacidad para reconocer intentos de phishing")
plt.xlabel("Respuesta")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("grafica_4_reconoce_phishing.png", bbox_inches="tight")
plt.show()


## Gráfica 5. Estado del antivirus

In [ ]:
# GRÁFICA 5: ESTADO DEL ANTIVIRUS

conteo = df_limpio["estado_antivirus"].value_counts().reset_index()
conteo.columns = ["Estado del antivirus", "Cantidad"]

display(conteo)

plt.figure(figsize=(10, 5))
plt.bar(conteo["Estado del antivirus"], conteo["Cantidad"])
plt.title("Uso y actualización del antivirus")
plt.xlabel("Estado")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig("grafica_5_estado_antivirus.png", bbox_inches="tight")
plt.show()


## Gráfica 6. Frecuencia de cambio de contraseñas

In [ ]:
# GRÁFICA 6: FRECUENCIA DE CAMBIO DE CONTRASEÑAS

conteo = df_limpio["cambio_contrasenas_anual"].value_counts().reset_index()
conteo.columns = ["Frecuencia", "Cantidad"]

display(conteo)

plt.figure(figsize=(10, 5))
plt.bar(conteo["Frecuencia"], conteo["Cantidad"])
plt.title("Frecuencia con la que cambian sus contraseñas")
plt.xlabel("Frecuencia anual")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig("grafica_6_cambio_contrasenas.png", bbox_inches="tight")
plt.show()


## Gráfica 7. Reutilización de contraseñas

In [ ]:
# GRÁFICA 7: REUTILIZACIÓN DE CONTRASEÑAS

conteo = df_limpio["reutiliza_contrasenas"].value_counts().reset_index()
conteo.columns = ["Respuesta", "Cantidad"]

display(conteo)

plt.figure(figsize=(8, 5))
plt.bar(conteo["Respuesta"], conteo["Cantidad"])
plt.title("Reutilización de contraseñas en varias cuentas")
plt.xlabel("Respuesta")
plt.ylabel("Cantidad de personas")
plt.tight_layout()
plt.savefig("grafica_7_reutiliza_contrasenas.png", bbox_inches="tight")
plt.show()


## Gráfica 8. Tipo de conexión a internet

In [ ]:
# GRÁFICA 8: TIPO DE CONEXIÓN A INTERNET

conteo = df_limpio["tipo_conexion"].value_counts().reset_index()
conteo.columns = ["Tipo de conexión", "Cantidad"]

display(conteo)

plt.figure(figsize=(10, 6))
plt.barh(conteo["Tipo de conexión"], conteo["Cantidad"])
plt.title("Tipo de conexión a internet utilizada")
plt.xlabel("Cantidad de personas")
plt.ylabel("Tipo de conexión")
plt.tight_layout()
plt.savefig("grafica_8_tipo_conexion.png", bbox_inches="tight")
plt.show()


## Gráfica 9. Relación entre conocimiento y reconocimiento de phishing

In [ ]:
# GRÁFICA 9: RELACIÓN ENTRE CONOCIMIENTO Y PHISHING

tabla_cruzada = pd.crosstab(
    df_limpio["nivel_conocimiento"],
    df_limpio["reconoce_phishing"]
)

display(tabla_cruzada)

tabla_cruzada.plot(kind="bar", figsize=(10, 6))
plt.title("Relación entre conocimiento y reconocimiento de phishing")
plt.xlabel("Nivel de conocimiento")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("grafica_9_conocimiento_vs_phishing.png", bbox_inches="tight")
plt.show()


## Gráfica 10. Promedio de manejo de ciberseguridad por nivel de conocimiento

In [ ]:
# GRÁFICA 10: PROMEDIO DE MANEJO POR NIVEL DE CONOCIMIENTO

promedios = (
    df_limpio
    .groupby("nivel_conocimiento")["manejo_ciberseguridad"]
    .mean()
    .reset_index()
    .sort_values("manejo_ciberseguridad", ascending=False)
)

display(promedios)

plt.figure(figsize=(10, 5))
plt.bar(promedios["nivel_conocimiento"], promedios["manejo_ciberseguridad"])
plt.title("Promedio de manejo de ciberseguridad por nivel de conocimiento")
plt.xlabel("Nivel de conocimiento")
plt.ylabel("Promedio de manejo")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("grafica_10_promedio_manejo.png", bbox_inches="tight")
plt.show()


## 6. Exportar CSV limpio

Se genera un archivo limpio que puede usarse en otros notebooks o como respaldo del análisis.


In [ ]:
# EXPORTAR CSV LIMPIO

nombre_salida = "encuesta_ciberseguridad_limpia.csv"

df_limpio.to_csv(
    nombre_salida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo exportado correctamente: {nombre_salida}")


## 7. Conclusiones sugeridas

Después de ejecutar el notebook, completá este análisis con tus resultados:

1. ¿Cuántas personas usan almacenamiento en la nube?
2. ¿Cuál es la plataforma de nube más utilizada?
3. ¿Qué nivel de conocimiento sobre ciberseguridad predomina?
4. ¿Qué tan bien reconocen los usuarios los intentos de phishing?
5. ¿Cuántas personas tienen antivirus actualizado?
6. ¿Con qué frecuencia cambian sus contraseñas?
7. ¿Cuántas personas reutilizan contraseñas?
8. ¿Qué tipo de conexión a internet predomina?
9. ¿Existe relación entre mayor conocimiento y mejor reconocimiento de phishing?
10. ¿Qué recomendaciones debería reforzar CyberLey con base en estos datos?

## Conclusión general

Este análisis permite utilizar datos históricos externos para complementar CyberLey.  
El sistema no solo recopila información desde la aplicación, sino que también puede procesar datos provenientes de encuestas externas, limpiarlos y convertirlos en información útil para la toma de decisiones.
